# Week 2 Session 1: Test-Driven Feature Engineering

Healthcare ML systems make decisions that directly affect patient outcomes. A subtle bug in a feature function -- for example, accidentally including *tomorrow's* lab result when computing *today's* risk score -- can make a model appear excellent in evaluation yet fail catastrophically in production. This class of bug, called **temporal leakage**, is the single most common and most dangerous failure mode in clinical ML pipelines.

In this session, you will learn a disciplined defense against these bugs: **test-driven feature engineering**. The methodology is simple but powerful:

> **For each feature:**
> 1. **Manually compute** the correct value from raw patient data
> 2. **Implement** the feature function in code
> 3. **Assert** that the computed value matches the manual calculation exactly

If the assertion fails, you have a bug. If it passes, you have a verified feature. No guessing, no "looks about right" -- just provable correctness.

---

### Learning Objectives

| # | Objective | Why It Matters |
|---|-----------|----------------|
| 1 | Apply test-driven development to feature engineering | Catches temporal leakage and off-by-one errors before they reach models |
| 2 | Implement 6 clinical features across 4 pattern types | Builds a reusable vocabulary of feature engineering patterns |
| 3 | Understand the cutoff date contract | The foundational concept for all temporal ML in healthcare |
| 4 | Generate classifier training data at scale | Creates the bridge from feature engineering to model training in Session 2 |

### Features You Will Build

| # | Feature | Pattern Type | Clinical Meaning |
|---|---------|-------------|------------------|
| 1 | `days_since_last_hba1c` | Temporal counter | How long since last diabetes monitoring -- identifies care gaps |
| 2 | `current_hba1c_level` | Clinical value | Most recent glycemic control reading (ADA target: < 7%) |
| 3 | `encounters_last_90d` | Rolling window | Recent healthcare utilization intensity |
| 4 | `age_at_date` | Demographic | Age-related complication risk |
| 5 | `current_systolic_bp` | Clinical value | Cardiovascular risk indicator (ADA target: < 130 mmHg) |
| 6 | `current_egfr` | Clinical value | Kidney function -- diabetes is the #1 cause of kidney failure |

These 6 features cover four distinct computational patterns. Once you recognize these patterns, adding new features (you will implement more in homework) becomes straightforward.

---

## Setup: Load EHR Data

We begin by loading the five core tables of our EHR dataset. These tables follow the standard relational model used in healthcare information systems:

| Table | Records | Contains |
|-------|---------|----------|
| **patients** | Demographics | Birthdate, gender, race, address |
| **encounters** | Visit records | Date, type (ED, inpatient, outpatient), reason |
| **conditions** | Diagnoses | ICD/SNOMED codes, description, onset date |
| **medications** | Prescriptions | Drug name, start/stop dates, dosage |
| **observations** | Lab results & vitals | HbA1c, blood pressure, eGFR, glucose, BMI |

The relational structure is: `PATIENTS --> ENCOUNTERS --> CONDITIONS, MEDICATIONS, OBSERVATIONS`. Every clinical event is linked to both a patient and an encounter.

## Setting Environment Up for Colab

Mount Google Drive and set the repository path so this notebook can access the EHR data and source code.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

### Project Configuration

This notebook loads paths from a YAML config file instead of hard-coded paths.

**Setup (one-time):**
1. In Colab's left sidebar, click the **Key** icon (Secrets)
2. Add a secret named `PROJECT_CONFIG_PATH`
3. Set the value to your config file path (e.g., `/content/drive/MyDrive/Project/config.yaml`)
4. Toggle "Notebook access" ON

In [ ]:
from google.colab import userdata
import yaml

try:
    config_path = userdata.get('PROJECT_CONFIG_PATH')
except:
    config_path = input("Enter path to your config.yaml: ")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

REPO_PATH = config['repo_path']
print(f"Repository path: {REPO_PATH}")

### Install Source Code as Package

`pip install` installs the local source code as a Python package so you can import directly from it (e.g., `from helpers import get_col`). Re-run this cell after making changes to the source code.

In [ ]:
!pip install {REPO_PATH}/src/ -q

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os, sys

from helpers import get_col
DATA_DIR = os.path.join(REPO_PATH, "data/week_1/processed_data/csv")

# Load all data tables
patients = pd.read_csv(os.path.join(DATA_DIR, "patients.csv"))
encounters = pd.read_csv(os.path.join(DATA_DIR, "encounters.csv"))
conditions = pd.read_csv(os.path.join(DATA_DIR, "conditions.csv"))
medications = pd.read_csv(os.path.join(DATA_DIR, "medications.csv"))
observations = pd.read_csv(os.path.join(DATA_DIR, "observations.csv"), low_memory=False)

print(f"Loaded:")
print(f"  Patients:      {len(patients):,}")
print(f"  Encounters:    {len(encounters):,}")
print(f"  Conditions:    {len(conditions):,}")
print(f"  Medications:   {len(medications):,}")
print(f"  Observations:  {len(observations):,}")

### Column Name Helper

EHR datasets from different sources may use different column naming conventions (`PATIENT` vs `patient` vs `Patient`). This helper function finds columns case-insensitively so our code works regardless of the naming convention in the data.

In [ ]:
patient_id_col = get_col(patients, 'id')
birthdate_col = get_col(patients, 'birthdate')
enc_patient_col = get_col(encounters, 'patient')
enc_date_col = get_col(encounters, 'date')
enc_desc_col = get_col(encounters, 'description')
obs_patient_col = get_col(observations, 'patient')
obs_desc_col = get_col(observations, 'description')
obs_value_col = get_col(observations, 'value')
obs_date_col = get_col(observations, 'date')
obs_code_col = get_col(observations, 'code')
med_patient_col = get_col(medications, 'patient')
med_start_col = get_col(medications, 'start')
cond_patient_col = get_col(conditions, 'patient')
cond_desc_col = get_col(conditions, 'description')

print("Column mappings configured")

### Encounter Classification

Our encounters table has a `DESCRIPTION` column but no standardized encounter class field. For downstream risk prediction, we need to distinguish between encounter types because **emergency department (ED) visits** and **inpatient hospitalizations** are the adverse events we will predict.

We derive the classification from description text:

| Encounter Class | Keywords | Clinical Meaning |
|----------------|----------|------------------|
| **emergency** | "emergency" | Unplanned ED visit -- often indicates acute crisis |
| **inpatient** | "inpatient", "admission to", "hospital admission" | Hospital stay -- the most resource-intensive and often most serious encounter |
| **outpatient** | "outpatient" | Scheduled clinic visit |
| **ambulatory** | (default) | Routine care, wellness visits, follow-ups |

This classification is critical: the `ExtendedPatientProfile` class (used later to generate training data) relies on the `ENCOUNTER_CLASS` column to identify ED visits and hospitalizations when computing risk labels.

In [ ]:
def classify_encounter(description):
    """Classify encounter type from description text."""
    desc = str(description).lower()
    if 'emergency' in desc:
        return 'emergency'
    elif any(kw in desc for kw in ['inpatient', 'admission to', 'hospital admission']):
        return 'inpatient'
    elif 'outpatient' in desc:
        return 'outpatient'
    else:
        return 'ambulatory'

encounters['ENCOUNTER_CLASS'] = encounters[enc_desc_col].apply(classify_encounter)

print("Encounter classification:")
print(encounters['ENCOUNTER_CLASS'].value_counts().to_string())

---

## Step 1 of TDD: Select a Sample Patient for Testing

The foundation of test-driven feature engineering is having a **known test case** where you can manually verify every computation. This requires selecting a sample patient whose data you can inspect by hand.

**Why not just test on random data?**

Random synthetic data lets you check that code *runs*, but not that it's *correct*. By selecting a real patient from our dataset, we can:
- Scroll through their actual HbA1c readings and count the days ourselves
- Look at their encounter dates and manually count visits in a 90-day window
- Confirm our function returns *exactly* the same number

**Selection criteria for a good test patient:**
- **Diabetic** -- we are building a diabetes risk prediction system
- **At least 10 encounters** -- enough visits to produce meaningful rolling window features
- **At least 3 HbA1c readings** -- enough lab data to test temporal counter and clinical value features
- **Ideally some variety** -- ED visits, hospitalizations, and routine visits in the mix

In [ ]:
# Diabetes-related terms for identification
DIABETES_TERMS = [
    'diabetes', 'diabetic', 'dm2', 'dm1', 't2dm', 't1dm',
    'type 2 diabetes', 'type 1 diabetes', 'diabetes mellitus'
]

def is_diabetic_condition(description):
    """Check if a condition description indicates diabetes."""
    if pd.isna(description):
        return False
    desc_lower = str(description).lower()
    return any(term in desc_lower for term in DIABETES_TERMS)

# Find diabetic patients
diabetic_from_csv = set(
    conditions[conditions[cond_desc_col].apply(is_diabetic_condition)][cond_patient_col].unique()
)

print(f"Diabetic patients identified: {len(diabetic_from_csv)}")

Now that we have the set of diabetic patients, we select a specific patient whose data we will manually inspect throughout this session.

In [ ]:
# We selected a diabetic patient with rich data across all feature types:
# - 57 encounters spanning several years
# - Multiple HbA1c, blood pressure, and eGFR readings before the midpoint
# - Encounters in the 90-day window around the midpoint
SAMPLE_PATIENT_ID = 'ff00fb3c-1c74-4ee5-88a4-2b57b3150b85'

n_enc = len(encounters[encounters[enc_patient_col] == SAMPLE_PATIENT_ID])
n_hba1c = len(observations[
    (observations[obs_patient_col] == SAMPLE_PATIENT_ID) &
    (observations[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
])

print(f"Selected sample patient for testing:")
print(f"  Patient ID: {SAMPLE_PATIENT_ID}")
print(f"  Encounters: {n_enc}")
print(f"  HbA1c readings: {n_hba1c}")

### Explore the Sample Patient's Timeline

Before computing any features, we examine the patient's encounter history to understand the data we are working with. We also select a **cutoff date** -- the simulated "today" that divides past (usable data) from future (what we predict).

Choosing the cutoff at the midpoint of the patient's timeline ensures we have enough data on both sides: enough history to compute meaningful features, and enough future events to check against.

In [ ]:
# Explore sample patient's timeline
patient_enc = encounters[encounters[enc_patient_col] == SAMPLE_PATIENT_ID].copy()
patient_enc['date'] = pd.to_datetime(patient_enc[enc_date_col], errors='coerce', utc=True)
patient_enc = patient_enc.sort_values('date')

print(f"Sample patient encounter timeline:")
print(f"  First encounter: {patient_enc['date'].min().date()}")
print(f"  Last encounter: {patient_enc['date'].max().date()}")
print(f"  Total encounters: {len(patient_enc)}")

# Choose cutoff in the middle of timeline
mid_idx = len(patient_enc) // 2
CUTOFF_DATE = patient_enc.iloc[mid_idx]['date']

print(f"\nSelected cutoff date: {CUTOFF_DATE.date()}")
print(f"  Encounters before cutoff: {mid_idx}")
print(f"  Encounters after cutoff: {len(patient_enc) - mid_idx}")

---

## The Cutoff Date Contract (Temporal Safety)

This is the most important concept in this entire session -- and arguably in the entire project.

### The Rule

> **Every feature function receives a cutoff date. It must use ONLY data on or before that date. Violating this rule is called temporal leakage.**

The cutoff date represents "right now" in a clinical prediction scenario. Imagine a nurse opens a patient's chart at 9 AM on March 15. A risk model should use only information available at that moment -- not lab results from March 20 that haven't happened yet.

### Visual Representation

```
                        CUTOFF DATE
                             |
    ◄── PAST ───────────────►|◄── FUTURE ──────────────►
                             |
    Features computed from   |  Label computed from
    data in this region      |  events in this region
    ONLY                     |  ONLY
                             |
    HbA1c readings           |  Did an ED visit or
    Encounter counts         |  hospitalization happen
    BP measurements          |  within 30 days?
    etc.                     |
```

### Why Temporal Leakage Is Dangerous

Unlike other ML bugs, leakage doesn't cause errors -- it causes **artificially good results**. A model that accidentally sees future data will:

| Metric | With Leakage | Without Leakage |
|--------|-------------|-----------------|
| AUC-ROC | 0.95+ | 0.65-0.80 |
| Recall | Very high | Moderate |
| Real-world performance | **Fails completely** | Matches evaluation |

The model appears to work brilliantly in evaluation, passes all your tests, gets deployed -- and then fails on real patients because it no longer has access to the future data it was secretly using.

### The Contract in Code

Every feature function in this session follows the same structure:

```python
def my_feature(patient_id, cutoff_date):
    # 1. Get patient's data
    patient_data = data[data['patient'] == patient_id]
    # 2. Filter to ON OR BEFORE cutoff (this is the safety line)
    before_cutoff = patient_data[patient_data['date'] <= cutoff_date]
    # 3. Compute feature from filtered data ONLY
    return compute_something(before_cutoff)
```

The `<=` filter on line 2 is the entire temporal safety mechanism. Every feature you build in this project must include this line.

---

## Feature Pattern Taxonomy

Before diving into individual features, it helps to see the big picture. Every temporal feature in healthcare ML falls into one of a few computational patterns. Once you internalize these patterns, you can design new features rapidly.

| Pattern | What It Computes | Pseudocode | Examples |
|---------|-----------------|------------|----------|
| **Temporal counter** | Days since last occurrence of an event | `cutoff - max(dates <= cutoff)` | Days since last HbA1c, days since last ED visit |
| **Clinical value** | Most recent measurement | `value at max(dates <= cutoff)` | Current HbA1c level, current BP, current eGFR |
| **Rolling window** | Count of events in a fixed lookback period | `count(events in [cutoff-N, cutoff])` | Encounters in last 90 days, ED visits in last 180 days |
| **Demographic** | Static or slowly changing patient attribute | `compute from demographics + cutoff` | Age at date, gender (static), years since diagnosis |

**Key insight:** The first three patterns all start with the same step -- *filter the timeline to before the cutoff date*. They differ only in what they compute from the filtered data. This shared structure is what makes temporal safety easy to enforce consistently.

We will implement one feature of each pattern type, plus two additional clinical value features for blood pressure and kidney function. The TDD cycle for each feature is identical:

```
Step 1: Manually compute TRUE value   --> "What should the answer be?"
Step 2: Implement the function         --> "Write code to compute it"
Step 3: Assert computed == TRUE        --> "Prove correctness"
```

---

## Feature 1: `days_since_last_hba1c` (Temporal Counter)

**Definition:** Number of days between the cutoff date and the most recent HbA1c lab reading on or before that date.

**Pattern:** Temporal counter -- `cutoff_date - max(hba1c_dates <= cutoff_date)`

**Clinical context:** HbA1c (glycated hemoglobin) is the primary biomarker for long-term blood glucose control. The American Diabetes Association (ADA) recommends testing every **3 months** (90 days) for patients not at target, and every **6 months** (180 days) for stable patients. When this feature exceeds 180 days, it signals a **care gap** -- the patient has fallen out of regular monitoring. Research consistently shows that care gaps correlate with worse glycemic outcomes and higher rates of complications.

**What to expect:**
- **0-90 days**: Actively monitored, testing on schedule
- **90-180 days**: Within guidelines for stable patients
- **180+ days**: Care gap -- patient may be disengaged from diabetes management
- **None**: No HbA1c on record before this date (new patient or very early in care)

### Step 1: Manually Compute the TRUE Value

We filter the patient's observation records to HbA1c readings on or before the cutoff, find the most recent one, and compute the date difference. This is the value our function *must* return.

In [ ]:
# Step 1: Manually compute TRUE value
patient_obs = observations[observations[obs_patient_col] == SAMPLE_PATIENT_ID].copy()
patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)

hba1c_before = patient_obs[
    (patient_obs['date'] <= CUTOFF_DATE) &
    (patient_obs[obs_desc_col].str.contains('hemoglobin a1c|hba1c', case=False, na=False))
].sort_values('date')

print(f"HbA1c readings BEFORE cutoff ({CUTOFF_DATE.date()}):")
for _, row in hba1c_before.iterrows():
    print(f"  {row['date'].date()}: {row[obs_value_col]}%")

if len(hba1c_before) > 0:
    last_hba1c_date = hba1c_before['date'].max()
    TRUE_days_since_hba1c = (CUTOFF_DATE - last_hba1c_date).days
    print(f"\nLast HbA1c: {last_hba1c_date.date()}")
    print(f"TRUE VALUE: {TRUE_days_since_hba1c} days")
else:
    TRUE_days_since_hba1c = None
    print("\nNo HbA1c before cutoff - TRUE VALUE: None")

### Step 2: Implement the Function

Now we write the function following the temporal safety contract. Notice the structure:
1. Get the patient's observations
2. **Filter to on or before cutoff** (the safety line)
3. Find the most recent HbA1c date
4. Return the difference in days

---

<details>
<summary><strong>Hint 1 — Functions to consider</strong> (click to expand)</summary>

`pd.to_datetime()`, `str.contains()`, `df.max()`, `df.idxmax()`, `timedelta.days`, `df.sort_values()`

*Not all of these are needed. Some may lead you down the wrong path.*
</details>

<details>
<summary><strong>Hint 2 — Conceptual direction</strong> (click to expand)</summary>

Filter the observations to find HbA1c readings — the description column contains the test name. After restricting to before the cutoff, you need the *most recent date*, not the most recent value. What's the difference between `.max()` and `.idxmax()`?
</details>

In [ ]:
# Step 2: Implement the function
def days_since_last_hba1c(patient_id, cutoff_date):
    """Days since most recent HbA1c reading on or before cutoff.

    Args:
        patient_id: The patient's unique identifier
        cutoff_date: The temporal boundary — only use data on or before this date

    Returns:
        int: Number of days since last HbA1c, or None if no readings exist before cutoff
    """
    # TODO: Get patient observations
    patient_obs = None

    # TODO: Filter to HbA1c readings on or before cutoff
    hba1c_before = None

    # TODO: Find most recent date and compute days difference
    result = None

    return result

# Test it
result = days_since_last_hba1c(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"days_since_last_hba1c = {result}")

### Step 3: Verify -- Does Computed Match TRUE?

This is the payoff of TDD. If the assertion passes, we have a mathematically verified feature. If it fails, we caught a bug before it could propagate into model training.

In [ ]:
# Step 3: Verify
computed = days_since_last_hba1c(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"Computed: {computed}")
print(f"Expected: {TRUE_days_since_hba1c}")
assert computed == TRUE_days_since_hba1c, f"MISMATCH! {computed} != {TRUE_days_since_hba1c}"
print("PASS")

---

## Feature 2: `current_hba1c_level` (Clinical Value)

**Definition:** The most recent HbA1c value (%) on or before the cutoff date.

**Pattern:** Clinical value -- `value at max(hba1c_dates <= cutoff_date)`

**Clinical context:** While Feature 1 tells us *when* the last HbA1c was measured, this feature tells us *what it said*. HbA1c reflects average blood glucose over the preceding 2-3 months (the lifespan of red blood cells). It is the single most important number in diabetes management.

The ADA classifies glycemic control as follows:

| HbA1c (%) | ADA Classification | Clinical Action |
|-----------|-------------------|-----------------|
| < 5.7 | Normal | No diabetes |
| 5.7 - 6.4 | Prediabetes | Lifestyle intervention, monitoring |
| 6.5 - 6.9 | Diabetes, at target | Continue current management |
| 7.0 - 8.9 | Above target | Intensify treatment, more frequent monitoring |
| 9.0 - 9.9 | Poor control | Urgent medication adjustment, consider referral |
| >= 10.0 | Very poor control | **High-risk event** in our labeling system |

> **Note:** An HbA1c >= 10% is one of the events that triggers the `will_have_high_risk_event_next_30d = 1` label in our training data.

### Step 1: Manually Compute the TRUE Value

In [ ]:
# Step 1: Manually compute TRUE value
# We already have hba1c_before from Feature 1
if len(hba1c_before) > 0:
    last_reading = hba1c_before.loc[hba1c_before['date'].idxmax()]
    TRUE_current_hba1c = float(last_reading[obs_value_col])
    print(f"Last HbA1c reading: {last_reading['date'].date()}")
    print(f"TRUE VALUE: {TRUE_current_hba1c}%")
else:
    TRUE_current_hba1c = None
    print("No HbA1c before cutoff - TRUE VALUE: None")

### Step 2: Implement the Function

This follows the same temporal safety pattern as Feature 1. The only difference is that instead of computing a *date difference*, we return the *value* at the most recent reading.

---

<details>
<summary><strong>Hint 1 — Functions to consider</strong> (click to expand)</summary>

`df.idxmax()`, `df.loc[]`, `float()`, `df.iloc[-1]`, `df.nlargest()`, `df.tail(1)`

*Not all of these are needed. Some may lead you down the wrong path.*
</details>

<details>
<summary><strong>Hint 2 — Conceptual direction</strong> (click to expand)</summary>

This is structurally identical to Feature 1 — same filter, same "find the most recent" logic. The only difference: instead of computing `cutoff - date`, you retrieve a *column value* from the row with the maximum date. How do you select a specific column from a row identified by its index?
</details>

In [ ]:
# Step 2: Implement the function
def current_hba1c_level(patient_id, cutoff_date):
    """Most recent HbA1c value on or before cutoff.

    Args:
        patient_id: The patient's unique identifier
        cutoff_date: The temporal boundary — only use data on or before this date

    Returns:
        float: Most recent HbA1c percentage, or None if no readings exist before cutoff
    """
    # TODO: Filter to HbA1c on or before cutoff (same as Feature 1)
    hba1c_before = None

    # TODO: Return the VALUE at the most recent reading
    result = None

    return result

# Test it
result = current_hba1c_level(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"current_hba1c_level = {result}")

### Step 3: Verify Against TRUE Value

In [ ]:
# Step 3: Verify
computed = current_hba1c_level(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"Computed: {computed}")
print(f"Expected: {TRUE_current_hba1c}")
assert computed == TRUE_current_hba1c, f"MISMATCH! {computed} != {TRUE_current_hba1c}"
print("PASS")

---

## Feature 3: `encounters_last_90d` (Rolling Window)

**Definition:** Number of healthcare encounters in the 90 days before (and including) the cutoff date.

**Pattern:** Rolling window -- `count(encounters where date in [cutoff - 90, cutoff])`

**Clinical context:** Healthcare utilization patterns carry strong predictive signal. This feature captures two distinct risk scenarios:

| Encounters in 90 Days | Interpretation | Risk Signal |
|-----------------------|----------------|-------------|
| 0 | No recent contact with healthcare system | Potential care gap -- patient may be disengaged |
| 1-3 | Normal follow-up schedule | Expected for stable diabetic patient |
| 4-6 | Elevated utilization | May indicate medication changes, new symptoms |
| 7+ | High utilization | Likely unstable -- frequent ER visits, acute episodes |

The 90-day window is a common choice in healthcare ML because it aligns with the typical quarterly follow-up cycle for chronic disease management. In homework, you will implement additional rolling windows (e.g., ED visits in last 180 days, hospitalizations in last 365 days) using this same pattern.

### Step 1: Manually Compute the TRUE Value

We define the 90-day window `[cutoff - 90 days, cutoff]`, filter encounters to within this window, and count them.

In [ ]:
# Step 1: Manually compute TRUE value
window_start = CUTOFF_DATE - timedelta(days=90)

enc_before = patient_enc[patient_enc['date'] <= CUTOFF_DATE]
enc_in_window = enc_before[(enc_before['date'] >= window_start) & (enc_before['date'] <= CUTOFF_DATE)]

print(f"90-day window: {window_start.date()} to {CUTOFF_DATE.date()}")
print(f"\nEncounters in window:")
for _, row in enc_in_window.iterrows():
    print(f"  {row['date'].date()}: {row[enc_desc_col][:50]}...")

TRUE_encounters_90d = len(enc_in_window)
print(f"\nTRUE VALUE: {TRUE_encounters_90d} encounters")

### Step 2: Implement the Function

The rolling window pattern differs from the temporal counter: instead of finding the *last* event and measuring distance, we define a *window* and count events inside it. Both patterns start by filtering to before the cutoff.

---

<details>
<summary><strong>Hint 1 — Functions to consider</strong> (click to expand)</summary>

`timedelta(days=...)`, `len()`, `pd.Series.between()`, `value_counts()`, `pd.to_datetime()`, `df.shape[0]`

*Not all of these are needed. Some may lead you down the wrong path.*
</details>

<details>
<summary><strong>Hint 2 — Conceptual direction</strong> (click to expand)</summary>

Unlike Features 1-2 which look for the *single most recent* event, this feature counts *all* events in a date range. Define the window boundaries first, then filter with two inequality conditions. The answer is the number of rows that survive the filter.
</details>

In [ ]:
# Step 2: Implement the function
def encounters_last_90d(patient_id, cutoff_date):
    """Count encounters in 90 days before cutoff.

    Args:
        patient_id: The patient's unique identifier
        cutoff_date: The temporal boundary — count encounters in [cutoff-90, cutoff]

    Returns:
        int: Number of encounters in the 90-day window
    """
    # TODO: Define the 90-day window
    window_start = None

    # TODO: Filter and count
    result = None

    return result

# Test it
result = encounters_last_90d(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"encounters_last_90d = {result}")

### Step 3: Verify Against TRUE Value

In [ ]:
# Step 3: Verify
computed = encounters_last_90d(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"Computed: {computed}")
print(f"Expected: {TRUE_encounters_90d}")
assert computed == TRUE_encounters_90d, f"MISMATCH! {computed} != {TRUE_encounters_90d}"
print("PASS")

---

## Feature 4: `age_at_date` (Demographic)

**Definition:** Patient's age in whole years at the cutoff date, computed from birthdate.

**Pattern:** Demographic -- `year(cutoff) - year(birthdate)`, adjusted for whether the birthday has occurred yet in the cutoff year.

**Clinical context:** Age is one of the strongest predictors of diabetes complications. The relationship is not simply "older = more risk" -- it interacts with other factors in clinically meaningful ways:

| Age Group | Key Risk Considerations |
|-----------|----------------------|
| < 40 | If Type 2, onset at young age indicates aggressive disease |
| 40-64 | Most common onset period for Type 2; cardiovascular risk rises |
| 65-74 | Higher complication rates; balance glycemic control vs. hypoglycemia risk |
| >= 75 | ADA recommends less aggressive HbA1c targets (< 8%) to avoid hypoglycemia |

**Implementation note:** Unlike the other features, `age_at_date` doesn't filter a timeline -- it computes from static demographics. However, it still depends on the cutoff date: a patient who is 64 in January and 65 in March has different risk profiles at those two dates.

### Step 1: Manually Compute the TRUE Value

In [ ]:
# Step 1: Manually compute TRUE value
patient_info = patients[patients[patient_id_col] == SAMPLE_PATIENT_ID].iloc[0]
birthdate = pd.to_datetime(patient_info[birthdate_col])

# Calculate age at cutoff
TRUE_age_at_date = CUTOFF_DATE.year - birthdate.year
# Adjust if birthday hasn't occurred yet this year
if (CUTOFF_DATE.month, CUTOFF_DATE.day) < (birthdate.month, birthdate.day):
    TRUE_age_at_date -= 1

print(f"Patient birthdate: {birthdate.date()}")
print(f"Cutoff date: {CUTOFF_DATE.date()}")
print(f"TRUE VALUE: {TRUE_age_at_date} years")

### Step 2: Implement the Function

Age calculation requires handling the birthday boundary: if the cutoff is before the patient's birthday this year, subtract one year.

---

<details>
<summary><strong>Hint 1 — Functions to consider</strong> (click to expand)</summary>

`pd.to_datetime()`, `dateutil.relativedelta()`, `.year`, `.month`, `.day`, `timedelta(days=365.25)`

*Not all of these are needed. Some may lead you down the wrong path.*
</details>

<details>
<summary><strong>Hint 2 — Conceptual direction</strong> (click to expand)</summary>

Start with the year difference: `cutoff_year - birth_year`. This overcounts by 1 for patients whose birthday hasn't happened yet in the cutoff year. How can you compare month-day pairs to check this?
</details>

In [ ]:
# Step 2: Implement the function
def age_at_date(patient_id, cutoff_date):
    """Patient's age in years at cutoff date.

    Args:
        patient_id: The patient's unique identifier
        cutoff_date: The date at which to compute age

    Returns:
        int: Age in whole years
    """
    # TODO: Calculate age, adjusting for birthday boundary
    result = None

    return result

# Test it
result = age_at_date(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"age_at_date = {result}")

### Step 3: Verify Against TRUE Value

In [ ]:
# Step 3: Verify
computed = age_at_date(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"Computed: {computed}")
print(f"Expected: {TRUE_age_at_date}")
assert computed == TRUE_age_at_date, f"MISMATCH! {computed} != {TRUE_age_at_date}"
print("PASS")

---

## The Diabetes-Hypertension-Kidney Triad

The next two features -- blood pressure and kidney function -- are not just "additional" metrics. They form a clinically critical triad with diabetes that drives treatment decisions:

```
         Diabetes
        /        \
       /          \
      v            v
 Hypertension <--> Kidney Disease
```

**How they interact:**
- **Diabetes damages kidneys** -- chronically elevated blood glucose injures the glomeruli (filtering units). Diabetes is the #1 cause of end-stage renal disease.
- **Diabetes causes hypertension** -- insulin resistance, vascular inflammation, and kidney damage all elevate blood pressure.
- **Hypertension accelerates kidney damage** -- high pressure in already-damaged glomeruli creates a vicious cycle of worsening filtration.
- **Kidney disease worsens hypertension** -- damaged kidneys cannot properly regulate fluid volume and sodium, further raising BP.

This is why diabetes management guidelines (ADA, KDIGO) treat these three conditions as an integrated problem. Our model needs all three signals -- hence these features.

### LOINC Codes

These features use **LOINC** (Logical Observation Identifiers Names and Codes), the international standard for identifying laboratory and clinical observations:

| Feature | LOINC Code | Observation Name |
|---------|-----------|-----------------|
| Systolic BP | `8480-6` | Systolic blood pressure |
| eGFR | `33914-3` | Glomerular filtration rate/1.73 sq M.predicted |

---

## Feature 5: `current_systolic_bp` (Clinical Value -- Blood Pressure)

**Definition:** Most recent systolic blood pressure reading (mmHg) on or before the cutoff date.

**Pattern:** Clinical value -- same as `current_hba1c_level`, but using LOINC code `8480-6` instead of text matching.

**Clinical context:** Systolic blood pressure (the "top number") measures the force of blood against artery walls when the heart beats. For diabetic patients, blood pressure control is as important as glycemic control:

| Systolic BP (mmHg) | ADA Classification | Clinical Action |
|--------------------|--------------------|-----------------|
| < 120 | Normal | No intervention needed |
| 120-129 | Elevated | Lifestyle modifications |
| 130-139 | Stage 1 hypertension | ADA target for most diabetic patients: < 130 |
| 140-179 | Stage 2 hypertension | Medication required; accelerates retinopathy and nephropathy |
| >= 180 | Hypertensive crisis | Immediate medical attention |

> **In Session 2**, a systolic BP > 140 will be one of the risk factors used in our clinical rules baseline model.

### Step 1: Manually Compute the TRUE Value

We use both the LOINC code (`8480-6`) and text matching (`"systolic"`) to find BP readings -- this dual approach handles datasets where codes may be inconsistently populated.

In [ ]:
# Step 1: Manually compute TRUE value for systolic BP
LOINC_SYSTOLIC_BP = '8480-6'

bp_before = patient_obs[
    (patient_obs['date'] <= CUTOFF_DATE) &
    (
        (patient_obs[obs_code_col].astype(str) == LOINC_SYSTOLIC_BP) |
        (patient_obs[obs_desc_col].str.lower().str.contains('systolic', na=False))
    )
].sort_values('date')

print(f"Systolic BP readings BEFORE cutoff ({CUTOFF_DATE.date()}):")
for _, row in bp_before.tail(5).iterrows():
    print(f"  {row['date'].date()}: {row[obs_value_col]} mmHg")

if len(bp_before) > 0:
    last_bp_reading = bp_before.loc[bp_before['date'].idxmax()]
    TRUE_current_systolic_bp = float(last_bp_reading[obs_value_col])
    print(f"\nLast systolic BP: {last_bp_reading['date'].date()}")
    print(f"TRUE VALUE: {TRUE_current_systolic_bp} mmHg")
else:
    TRUE_current_systolic_bp = None
    print("\nNo systolic BP before cutoff - TRUE VALUE: None")

### Step 2: Implement the Function

Same clinical value pattern as `current_hba1c_level` -- filter to before cutoff, find most recent, return the value. The only difference is the observation identifier (LOINC code + text fallback).

---

<details>
<summary><strong>Hint 1 — Functions to consider</strong> (click to expand)</summary>

`obs_code_col`, `str.contains()`, `.astype(str)`, `float()`, `df.query()`, `|` (OR operator)

*Not all of these are needed.*
</details>

<details>
<summary><strong>Hint 2 — Conceptual direction</strong> (click to expand)</summary>

You've already implemented the clinical value pattern in Feature 2. Apply the same logic here, but observations can be identified in two ways: by LOINC code *or* by description text. Using both makes your code robust to inconsistent data. The LOINC code for systolic BP is `8480-6`.
</details>

In [ ]:
# Step 2: Implement the function
def current_systolic_bp(patient_id, cutoff_date):
    """Most recent systolic BP value on or before cutoff. LOINC: 8480-6"""
    # TODO: Define the LOINC code for systolic BP
    LOINC_SYSTOLIC_BP = None

    # TODO: Get patient observations and parse dates to datetime (UTC)
    patient_obs = None

    # TODO: Filter to before cutoff AND matching systolic BP (by LOINC code OR description text)
    bp_before = None

    # TODO: Handle empty case — return None if no readings exist

    # TODO: Get the value at the most recent reading date
    last_value = None

    return float(last_value)

# Test it
result = current_systolic_bp(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"current_systolic_bp = {result}")

### Step 3: Verify Against TRUE Value

In [ ]:
# Step 3: Verify
computed = current_systolic_bp(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"Computed: {computed}")
print(f"Expected: {TRUE_current_systolic_bp}")
assert computed == TRUE_current_systolic_bp, f"MISMATCH! {computed} != {TRUE_current_systolic_bp}"
print("PASS")

---

## Feature 6: `current_egfr` (Clinical Value -- Kidney Function)

**Definition:** Most recent eGFR (estimated Glomerular Filtration Rate) reading on or before the cutoff date, measured in mL/min/1.73m2.

**Pattern:** Clinical value -- same pattern as Features 2 and 5, using LOINC code `33914-3`.

**Clinical context:** eGFR estimates how efficiently the kidneys filter waste from the blood. It is calculated from serum creatinine, age, and sex using the CKD-EPI equation. For diabetic patients, tracking eGFR over time is critical because kidney damage from diabetes is often **silent** until advanced stages.

The KDIGO (Kidney Disease: Improving Global Outcomes) staging system:

| eGFR (mL/min/1.73m2) | CKD Stage | Clinical Interpretation | Action |
|----------------------|-----------|------------------------|--------|
| >= 90 | G1 | Normal or high | Monitor annually if diabetes present |
| 60-89 | G2 | Mildly decreased | Monitor every 6-12 months |
| 45-59 | G3a | Mild-to-moderate | Refer to nephrology, adjust medications |
| 30-44 | G3b | Moderate-to-severe | Active nephrology management |
| 15-29 | G4 | Severely decreased | Prepare for renal replacement therapy |
| < 15 | G5 | Kidney failure | Dialysis or transplant needed |

> **In Session 2**, an eGFR < 60 (CKD stage 3+) will be one of the risk factors used in our clinical rules baseline model.

**Key clinical fact:** A patient can lose up to 50% of kidney function (eGFR dropping from 90 to 45) before experiencing any symptoms. This is why regular monitoring and predictive models are so important -- by the time symptoms appear, significant irreversible damage has occurred.

### Step 1: Manually Compute the TRUE Value

In [ ]:
# Step 1: Manually compute TRUE value for eGFR
LOINC_EGFR = '33914-3'

egfr_before = patient_obs[
    (patient_obs['date'] <= CUTOFF_DATE) &
    (
        (patient_obs[obs_code_col].astype(str) == LOINC_EGFR) |
        (patient_obs[obs_desc_col].str.lower().str.contains('glomerular|egfr', na=False))
    )
].sort_values('date')

print(f"eGFR readings BEFORE cutoff ({CUTOFF_DATE.date()}):")
for _, row in egfr_before.tail(5).iterrows():
    print(f"  {row['date'].date()}: {row[obs_value_col]} mL/min/1.73m2")

if len(egfr_before) > 0:
    last_egfr_reading = egfr_before.loc[egfr_before['date'].idxmax()]
    TRUE_current_egfr = float(last_egfr_reading[obs_value_col])
    print(f"\nLast eGFR: {last_egfr_reading['date'].date()}")
    print(f"TRUE VALUE: {TRUE_current_egfr} mL/min/1.73m2")
else:
    TRUE_current_egfr = None
    print("\nNo eGFR before cutoff - TRUE VALUE: None")

### Step 2: Implement the Function

Identical pattern to `current_systolic_bp` -- only the LOINC code and text filter change.

---

<details>
<summary><strong>Hint 1 — Functions to consider</strong> (click to expand)</summary>

`obs_code_col`, `str.contains()`, `|`, `float()`, `.astype(str)`, `df.idxmax()`
</details>

<details>
<summary><strong>Hint 2 — Conceptual direction</strong> (click to expand)</summary>

Same clinical value pattern as Features 2 and 5. LOINC code: `33914-3`. For the text fallback, think about what medical terms describe kidney filtration rate.
</details>

In [ ]:
# Step 2: Implement the function
def current_egfr(patient_id, cutoff_date):
    """Most recent eGFR value on or before cutoff. LOINC: 33914-3"""
    # TODO: Define the LOINC code for eGFR
    LOINC_EGFR = None

    # TODO: Get patient observations and parse dates to datetime (UTC)
    patient_obs = None

    # TODO: Filter to before cutoff AND matching eGFR (by LOINC code OR description text)
    egfr_before = None

    # TODO: Handle empty case — return None if no readings exist

    # TODO: Get the value at the most recent reading date
    last_value = None

    return float(last_value)

# Test it
result = current_egfr(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"current_egfr = {result}")

### Step 3: Verify Against TRUE Value

In [ ]:
# Step 3: Verify
computed = current_egfr(SAMPLE_PATIENT_ID, CUTOFF_DATE)
print(f"Computed: {computed}")
print(f"Expected: {TRUE_current_egfr}")
assert computed == TRUE_current_egfr, f"MISMATCH! {computed} != {TRUE_current_egfr}"
print("PASS")

### Feature Verification Complete

All 6 features pass the TDD verification:

| Feature | Pattern | Verified |
|---------|---------|----------|
| `days_since_last_hba1c` | Temporal counter | PASS |
| `current_hba1c_level` | Clinical value | PASS |
| `encounters_last_90d` | Rolling window | PASS |
| `age_at_date` | Demographic | PASS |
| `current_systolic_bp` | Clinical value (LOINC) | PASS |
| `current_egfr` | Clinical value (LOINC) | PASS |

**What did we prove?** Each function returns exactly the value we computed by hand from raw data. More importantly, each function respects temporal safety -- it only uses data on or before the cutoff date.

**What did we demonstrate about patterns?** The clinical value pattern (Features 2, 5, 6) is structurally identical code with different observation filters. This is the power of recognizing feature patterns -- you can now add `current_diastolic_bp`, `current_bmi`, or any other observation-based feature by changing one line.

---

## From Features to Training Data: Scaling Up

So far, we have built and verified 6 feature functions on a single patient at a single cutoff date. To train machine learning models in Session 2, we need to compute these features for **every diabetic patient** at **many cutoff dates**, each paired with a label.

### The Labeling Strategy

For each (patient, cutoff_date) pair, we compute a binary label:

| Label | Name | Definition |
|-------|------|------------|
| **1** | High-risk | At least one of these occurs within 30 days *after* the cutoff: diabetes-related ED visit, diabetes-related hospitalization, or HbA1c >= 10% |
| **0** | Low-risk | None of the above events occur within 30 days after the cutoff |

This is the **ADA 2025 definition** of a high-risk event for diabetic patients.

### Temporal Safety in the Label

Notice that the label uses only events *after* the cutoff, while features use only data *on or before* the cutoff. The feature-label boundary is airtight:

```
Features computed HERE          Label computed HERE
         |                              |
    [past data] ◄── cutoff ──► [next 30 days]
```

### Why Use `ExtendedPatientProfile`?

Computing features by repeatedly filtering DataFrames (as we did above for one patient) is fine for verification, but too slow for 2,000+ patients. The `ExtendedPatientProfile` class:

- **Pre-indexes** all patient data into sorted in-memory timelines during loading
- **Pre-parses** all dates once (not repeatedly in each feature call)
- Provides `generate_all_instances_session_1()` which computes the same 6 features we verified, plus the label, at regular intervals across the patient's timeline

This makes generation ~100x faster than the DataFrame-filtering approach.

In [ ]:
import importlib

try:
    import tqdm
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'tqdm', '-q'])

from profiles.extended_patient_profile import ExtendedPatientProfile

print(f"ExtendedPatientProfile loaded")
print(f"  load_from_dataframes: available")
print(f"  generate_all_instances_session_1: available")

### Load Patient Profiles

We load all diabetic patient data into `ExtendedPatientProfile` objects, which pre-index timelines for fast feature computation.

In [ ]:
# Load patient profiles from DataFrames (diabetic patients only)
# Note: we pass the encounters DataFrame WITH the ENCOUNTER_CLASS column
# so that ED visits and hospitalizations are correctly classified
profiles = ExtendedPatientProfile.load_from_dataframes(
    patients_df=patients,
    encounters_df=encounters,
    conditions_df=conditions,
    observations_df=observations,
    medications_df=medications,
    diabetic_only=True,
    show_progress=True
)

print(f"\nExample profile: {list(profiles.values())[0]}")

### Generate Training Instances

For each patient, we generate (cutoff_date, features, label) instances at weekly intervals across their timeline.

In [ ]:
# Generate training data using profile.generate_all_instances_session_1()
from tqdm import tqdm

print("Generating classifier training data with 6 Session 1 features...")
print("(Using weekly intervals to keep dataset size practical)")

all_instances = []
for profile in tqdm(profiles.values(), desc="Processing patients"):
    instances = profile.generate_all_instances_session_1(interval_days=7)
    all_instances.extend(instances)

classifier_df = pd.DataFrame(all_instances)
n_contributing = classifier_df['patient_id'].nunique()
print(f"\nGenerated {len(classifier_df):,} instances from {n_contributing:,} patients ({len(profiles):,} profiles loaded)")
print(f"Columns: {list(classifier_df.columns)}")

### Inspect the Dataset

In [ ]:
# Examine the generated dataset
print("Dataset Statistics:")
print(f"  Total instances: {len(classifier_df):,}")
print(f"  Unique patients: {classifier_df['patient_id'].nunique():,}")
print(f"  Date range: {classifier_df['date'].min()} to {classifier_df['date'].max()}")

target_col = 'will_have_high_risk_event_next_30d'
print(f"\nClass Distribution:")
print(classifier_df[target_col].value_counts().to_string())
print(f"\nPositive rate: {classifier_df[target_col].mean():.2%}")

### Interpreting the Class Imbalance

The positive rate of ~2-3% means that for every high-risk instance, there are roughly 35 low-risk instances. This **extreme class imbalance** is typical of clinical prediction tasks -- most patients on most days are *not* about to have an adverse event.

This imbalance has direct consequences for model training in Session 2:

| Challenge | Why It Matters | How We Will Address It |
|-----------|---------------|----------------------|
| A "predict 0 always" model gets ~97% accuracy | Accuracy is misleading with imbalanced data | Use AUC-ROC, precision, recall instead of accuracy |
| Models tend to ignore the minority class | Few positive examples to learn from | Use `class_weight='balanced'` in logistic regression |
| Random oversampling can leak information | Duplicated samples may appear in train and test | Use patient-level splitting, not row-level |

Understanding this imbalance now will help you interpret Session 2's baseline results correctly.

In [ ]:
# Save classifier training data
output_dir = os.path.join(REPO_PATH, "data/week_2")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "classifier_training_data.csv")
classifier_df.to_csv(output_path, index=False)

feature_cols = [c for c in classifier_df.columns
                if c not in ['patient_id', 'date', 'will_have_high_risk_event_next_30d']]

print(f"Classifier training data saved to: {output_path}")
print(f"  Shape: {classifier_df.shape[0]:,} rows x {classifier_df.shape[1]} columns")
print(f"\nFeatures: {feature_cols}")
print(f"Target: will_have_high_risk_event_next_30d")

---

## Summary

### What You Accomplished

1. **Learned test-driven feature engineering** -- a methodology where every feature is verified against a manually computed ground truth before being used in model training

2. **Implemented and verified 6 features** across 4 computational patterns:

| Feature | Pattern | What It Captures |
|---------|---------|-----------------|
| `days_since_last_hba1c` | Temporal counter | Care gap detection -- is this patient being monitored? |
| `current_hba1c_level` | Clinical value | Glycemic control status -- how is diabetes being managed? |
| `encounters_last_90d` | Rolling window | Utilization intensity -- stable care or crisis pattern? |
| `age_at_date` | Demographic | Age-related risk adjustment |
| `current_systolic_bp` | Clinical value (LOINC) | Cardiovascular risk -- is hypertension accelerating complications? |
| `current_egfr` | Clinical value (LOINC) | Kidney function -- is diabetic nephropathy progressing? |

3. **Understood the diabetes-hypertension-kidney triad** and why our feature set covers all three axes of diabetic complication risk

4. **Generated classifier training data** for all diabetic patients with binary labels for 30-day adverse event prediction (ADA 2025 definition: ED visit, hospitalization, or HbA1c >= 10%)

### Key Takeaways

**Temporal safety** is not a suggestion -- it is a hard requirement. Every feature filters data to `<= cutoff_date`. The label uses only events `> cutoff_date`. Violating this causes leakage that produces misleadingly good evaluation metrics but real-world failure.

**Pattern recognition accelerates feature engineering.** The clinical value pattern (Features 2, 5, 6) is structurally identical code with different observation filters. Once you recognize this, adding new features becomes a matter of identifying the right LOINC code or text filter.

**Class imbalance is expected.** A ~97:3 negative-to-positive ratio is normal for clinical event prediction. This shapes every modeling decision in Session 2.

---

## Next Steps

In **Session 2: Baseline Models**, you will:
- Load the training data generated above
- Implement **patient-level data splitting** (preventing leakage from correlated observations of the same patient appearing in both train and test)
- Build **5 baseline models** of increasing sophistication:
  1. Random prediction (the floor)
  2. Majority class (what "always predict 0" achieves)
  3. Single-feature threshold (HbA1c > 9%)
  4. Clinical rules (multi-factor risk counting)
  5. Logistic regression (first real ML model)
- Establish the **performance benchmarks** that any more complex model must beat

---

## Appendix: From Notebook Functions to Class Methods

In the homework, you will implement feature methods in `ExtendedPatientProfile`. This section explains how to transform the notebook functions you wrote above into class methods.

### The Key Transformation

| Aspect | Notebook Function | Class Method |
|--------|------------------|--------------|
| **Parameters** | `patient_id`, `cutoff_date` | `self`, `target_date` |
| **Data source** | Filter global DataFrame by `patient_id` | Use `self.<timeline>` (already patient-specific) |
| **Date parsing** | Parse dates in each call | Already parsed during profile loading |
| **Filtering** | DataFrame boolean indexing | List comprehension |
| **Find max date** | `idxmax()` on DataFrame | `max()` with key function |

### Why the Difference?

In the notebook, you worked with **global DataFrames** containing all patients. Each function had to:
1. Filter to one patient's data
2. Parse date strings to datetime objects
3. Apply the temporal filter

In the class, the `ExtendedPatientProfile` object already has:
1. **Patient-specific timelines** (`self.hba1c_timeline`, `self.encounter_timeline`, etc.)
2. **Pre-parsed dates** (done once during `load_from_dataframes`)
3. **Sorted data** (timelines are kept sorted by date)

### Example: `current_egfr`

**Notebook function (DataFrame-based):**
```python
def current_egfr(patient_id, cutoff_date):
    """Most recent eGFR value on or before cutoff."""
    LOINC_EGFR = '33914-3'
    
    # Get this patient's observations
    patient_obs = observations[observations[obs_patient_col] == patient_id].copy()
    patient_obs['date'] = pd.to_datetime(patient_obs[obs_date_col], utc=True)
    
    # Filter to eGFR readings before cutoff
    egfr_before = patient_obs[
        (patient_obs['date'] <= cutoff_date) &
        (
            (patient_obs[obs_code_col].astype(str) == LOINC_EGFR) |
            (patient_obs[obs_desc_col].str.lower().str.contains('glomerular|egfr', na=False))
        )
    ]
    
    if len(egfr_before) == 0:
        return None
    
    last_value = egfr_before.loc[egfr_before['date'].idxmax(), obs_value_col]
    return float(last_value)
```

**Class method (timeline-based):**
```python
def _get_most_recent_egfr(self, target_date: datetime) -> Optional[float]:
    """Most recent eGFR value before target_date."""
    # Timeline is already patient-specific and dates are pre-parsed
    readings_before = [
        r for r in self.egfr_timeline
        if safe_parse_date(r['date']) and safe_parse_date(r['date']) <= target_date
    ]
    
    if not readings_before:
        return None
    
    last_reading = max(readings_before, key=lambda x: safe_parse_date(x['date']))
    return last_reading.get('value')
```

### Key Simplifications

1. **No `patient_id` parameter** -- the method operates on `self`, which is already one patient's profile

2. **No DataFrame filtering** -- `self.egfr_timeline` contains only this patient's eGFR readings

3. **No LOINC/text matching** -- the timeline was populated during loading, so it only contains the right observations

4. **List comprehension instead of boolean indexing** -- cleaner syntax for filtering a list of dicts

### The Pattern

For any notebook function, the class method transformation follows this template:

```python
# NOTEBOOK
def feature_name(patient_id, cutoff_date):
    patient_data = dataframe[dataframe['patient'] == patient_id]
    patient_data['date'] = pd.to_datetime(patient_data['date'])
    before_cutoff = patient_data[patient_data['date'] <= cutoff_date]
    # ... compute from before_cutoff ...

# CLASS METHOD
def _feature_name(self, target_date: datetime) -> <ReturnType>:
    before_cutoff = [
        e for e in self.<timeline>
        if safe_parse_date(e['date']) and safe_parse_date(e['date']) <= target_date
    ]
    # ... compute from before_cutoff ...
```

### Available Timelines in ExtendedPatientProfile

| Timeline | Contents | Use For |
|----------|----------|---------|
| `self.encounter_timeline` | All encounters | `days_since_last_encounter`, `encounters_last_90d` |
| `self.medication_timeline` | Medication start/stop events | `days_since_medication_change`, `active_medication_count` |
| `self.emergency_visit_timeline` | ED visits | `emergency_visits_last_180d` |
| `self.hospitalization_timeline` | Inpatient stays | `hospitalizations_last_365d` |
| `self.hba1c_timeline` | HbA1c readings | `current_hba1c_level`, `hba1c_trend` |
| `self.bp_timeline` | Blood pressure | `current_systolic_bp`, `bp_trend` |
| `self.egfr_timeline` | eGFR readings | `current_egfr`, `egfr_trend` |
| `self.bmi_timeline` | BMI readings | `bmi_category` |
| `self.care_gap_timeline` | Care gaps | `longest_care_gap_days` |

### Helper Methods You Can Use

The class provides two generic helpers that work with any timeline:

```python
# Days since last event in any timeline
self._days_since_last_event(target_date, self.encounter_timeline)

# Count events in a window for any timeline
self._count_events_in_window(target_date, self.medication_timeline, 90)
```

Many homework features are one-liners using these helpers!